In [1]:
# 모듈 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('/content/drive/MyDrive/KDThome/santander-customer-transaction-prediction/train.csv')
df.head()

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [3]:
df_1 = df.drop('ID_code', axis=1, inplace=False)

In [5]:
# 전처리는 이상치만. 결측치는 없음
# 소수의 이상치는 중앙값으로
# target 제외한 컬럼만 추출
features = df_1.drop('target', axis=1)

# 이상치를 중앙값으로 대체
df_fixed = df_1.copy()

for col in features.columns:
    Q1 = df_1[col].quantile(0.25)
    Q3 = df_1[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 이상치 탐지
    outlier_mask = (df_1[col] < lower_bound) | (df_1[col] > upper_bound)

    # 이상치 중앙값으로 대체
    median = df_1[col].median()
    df_fixed.loc[outlier_mask, col] = median

# 결과 확인
print("✅ 이상치 중앙값으로 대체 완료")
print(df_fixed.shape)

✅ 이상치 중앙값으로 대체 완료
(200000, 201)


In [7]:
# 데이터분리
x_data = df_fixed.drop('target', axis=1, inplace=False).values
t_data = df_fixed['target'].values

# 테스트 데이터 분리
x_data_train, x_data_test, t_data_train, t_data_test =\
train_test_split(x_data,
                 t_data,
                 test_size=0.2,
                 stratify=t_data)

In [6]:
df_fixed.head()

,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,-4.9200,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,3.1468,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,-4.9193,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,-5.8609,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,6.2654,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [8]:
# 정규화
scaler = StandardScaler()
x_data_train_norm = scaler.fit_transform(x_data_train)
x_data_test_norm = scaler.fit_transform(x_data_test)

In [9]:
# 오버샘플링
sm = SMOTE()
x_data_train_norm_sm, t_data_train_sm = sm.fit_resample(x_data_train_norm, t_data_train)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

# 로지스틱 회귀 모델
logistic = LogisticRegression()

In [14]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.1 MB/s eta 0:00:00


In [15]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold


In [16]:
# Optuna 최적화용 함수
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'use_label_encoder': False,
        'verbosity': 0,
        'n_jobs': -1
    }

    model = XGBClassifier(**params)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, x_data_train_norm_sm, t_data_train_sm,
                             scoring='f1', cv=skf, n_jobs=-1)
    return scores.mean()

In [17]:
# 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# 최적 파라미터 출력
print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

[I 2025-04-18 08:08:41,592] A new study created in memory with name: no-name-abd16dfa-a680-410b-8648-1bec2b2be467
[I 2025-04-18 08:18:14,333] Trial 0 finished with value: 0.9443054830744595 and parameters: {'n_estimators': 436, 'max_depth': 10, 'learning_rate': 0.12900066274252145, 'subsample': 0.7766380612503516, 'colsample_bytree': 0.9325177961785522, 'gamma': 3.642410999980977, 'reg_alpha': 0.9377715474184656, 'reg_lambda': 0.612426567669206}. Best is trial 0 with value: 0.9443054830744595.
[I 2025-04-18 08:19:41,054] Trial 1 finished with value: 0.792118502452547 and parameters: {'n_estimators': 178, 'max_depth': 3, 'learning_rate': 0.016045308599646528, 'subsample': 0.5386612177484387, 'colsample_bytree': 0.6198613331525775, 'gamma': 2.057203725408553, 'reg_alpha': 0.4677886931601777, 'reg_lambda': 0.47256347494232587}. Best is trial 0 with value: 0.9443054830744595.
[I 2025-04-18 08:21:50,349] Trial 2 finished with value: 0.8696510523062851 and parameters: {'n_estimators': 119, '

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
best_params.update({
    'use_label_encoder': False,
    'verbosity': 0,
    'n_jobs': -1
})

xgb_best = XGBClassifier(**best_params)
xgb_best.fit(x_data_train_norm_sm, t_data_train_sm)


In [ ]:
# 로지스틱 회귀 모델 정의
logistic = LogisticRegression(max_iter=1000, random_state=42)

# 최적 XGBoost 모델 정의
best_params.update({
    'use_label_encoder': False,
    'verbosity': 0,
    'n_jobs': -1
})
xgb_best = XGBClassifier(**best_params)

# 앙상블 모델 (soft voting 사용)
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_best),
        ('logistic', logistic)
    ],
    voting='soft',  # 확률 평균 기반
    n_jobs=-1
)

In [ ]:
# 학습
ensemble_model.fit(x_data_train_norm_sm, t_data_train_sm)